In [1]:
import sys
import os
import pandas as pd
import numpy as np

# Make project root importable and set as reference point for paths
sys.path.append("..")

# Build the path relative to the project root, not the notebook's location
DATA_PATH = os.path.join("..", "data", "raw", "mental-heath-in-tech-2016_20161114.csv")

df = pd.read_csv(DATA_PATH)

In [2]:
from src import feature_config
from src.preprocessing import run_pipeline

#df_processed = run_pipeline(df, feature_config)

In [3]:
# For chi-square / other feature-selection diagnostics:
from src.feature_selection import apply_chi_square, apply_correlation_filter, apply_mutual_information, apply_variance_threshold
from src.preprocessing import impute_structural_missingness, prepare_for_feature_selection, fill_remaining_with_mean

#df_selection_ready = prepare_for_feature_selection(df, feature_config)
#chi_square_results = apply_chi_square(df_selection_ready, feature_config.FEATURE_FILTER_GROUPS)
#mi_results = apply_mutual_information(df_selection_ready, feature_config.FEATURE_FILTER_GROUPS)

# For the actual final model-ready matrix (unchanged usage):
df_processed = run_pipeline(df, feature_config)

df_imputed = impute_structural_missingness(df_processed, feature_config.STRUCTURAL_MISSINGNESS)
df_imputed = fill_remaining_with_mean(df_imputed, exclude_cols=["age_group"])

#variance_threshold_results = apply_variance_threshold(df_imputed)
corr_matrix, high_corr_features = apply_correlation_filter(df_imputed, threshold=0.7)


negative_impact_reveal: 75 rows answered despite reveal_to_clients_direction==2
negative_impact_reveal_coworker: 30 rows answered despite reveal_to_coworkers_direction==2
percentage_affected: 38 rows answered despite productivity_affected==1


In [7]:
columns_to_drop_variance = variance_threshold_results.loc[
    variance_threshold_results["kept"] == False, "feature"
].tolist()

# Override: keep reveal_to_clients_direction despite failing the threshold --
# its low post-imputation variance is a mean-imputation artifact, not a genuine
# lack of information (see DECISIONS.md)
columns_to_drop_variance.remove("reveal_to_clients_direction")

print(len(columns_to_drop_variance), "columns to drop (variance):")
print(columns_to_drop_variance)

26 columns to drop (variance):
['percentage_affected_not_applicable', 'believed_conditions__burn_out', 'diagnosed_conditions__other', 'diagnosed_conditions__sleeping_disorder', 'believed_conditions__gender_dysphoria', 'diagnosed_conditions_professional__burn_out', 'diagnosed_conditions__burn_out', 'believed_conditions__depersonalization_disorder', 'diagnosed_conditions_professional__gender_dysphoria', 'diagnosed_conditions__gender_dysphoria', 'believed_conditions__other', 'believed_conditions__autism', 'reason_not_willing_physical_health__embarrassment_shame', 'believed_conditions__dissociative_disorder', 'diagnosed_conditions__psychotic_disorder', 'reason_not_willing_mental_health__embarrassment_shame', 'diagnosed_conditions_professional__psychotic_disorder', 'believed_conditions__psychotic_disorder', 'diagnosed_conditions__autism', 'reason_not_willing_mental_health__timing_later', 'diagnosed_conditions_professional__autism', 'diagnosed_conditions_professional__dissociative_disorder',

In [4]:
high_corr_features

,feature_a,feature_b,correlation
9182,diagnosed_conditions__burn_out,diagnosed_conditions_professional__burn_out,1.000000
25438,gender_cleaned_male,gender_cleaned_female,-0.926514
30013,reveal_to_clients_special_not_applicable_to_me,reveal_to_coworkers_special_not_applicable_to_me,0.909509
32758,negative_impact_reveal_not_applicable,negative_impact_reveal_coworker_not_applicable,0.828328
32392,interferes_with_work_treated_special_not_appli...,interferes_with_work_not_treated_special_not_a...,0.824498
8816,diagnosed_conditions__attention_deficit_hypera...,diagnosed_conditions_professional__attention_d...,0.798425
7184,diagnosed_by_professional,diagnosed_conditions_professional__mood_disorder,0.766650
30196,reveal_to_coworkers_special_not_applicable_to_me,negative_impact_reveal_coworker_special_not_ap...,0.761980
10097,diagnosed_conditions__obsessive_compulsive_dis...,diagnosed_conditions_professional__obsessive_c...,0.746790
6773,past_mental_health_disorder,diagnosed_by_professional,0.743473


In [20]:
ct = pd.crosstab(df_selection_ready['mental_health_benefits'], df_selection_ready['previous_employers_mental_health_benefits'])
total = ct.values.sum()
concordant = np.trace(ct.values) if ct.shape[0] == ct.shape[1] else None
pct = f"{concordant/total*100:.1f}%" if concordant is not None else "n/a (unequal categories)"
print(f"concordance = {pct} (n={total})")
print(ct)

concordance = n/a (unequal categories) (n=757)
previous_employers_mental_health_benefits  0.0  1.0  2.0
mental_health_benefits                                  
I don't know                                17   76   70
No                                          11   46  104
Not eligible for coverage / N/A              6    8   31
Yes                                        117  196   75
